<a href="https://colab.research.google.com/github/PatriciaKiarie04/bayesian-classification-diabetes/blob/main/Bayesian_group_work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/diabetes.csv')
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if 'diabetes' in f.lower() or f.lower().endswith('.csv'):
            print(os.path.join(root, f))

/content/drive/MyDrive/Reviews.csv
/content/drive/MyDrive/diabetes.csv
/content/drive/MyDrive/Colab Notebooks/sentiment_analysis_nlp.csv


In [8]:
zero_check_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_check_cols:
    n_zeros = (df[col] == 0).sum()
    print(f"{col}: {n_zeros} zero values ({n_zeros/len(df)*100:.1f}%)")

Glucose: 5 zero values (0.7%)
BloodPressure: 35 zero values (4.6%)
SkinThickness: 227 zero values (29.6%)
Insulin: 374 zero values (48.7%)
BMI: 11 zero values (1.4%)


In [9]:
# Treat biologically impossible zeros as missing
impute_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[impute_cols] = df[impute_cols].replace(0, np.nan)

# Impute using the median WITHIN each outcome class (0 = no diabetes, 1 = diabetes)
for col in impute_cols:
    df[col] = df.groupby('Outcome')[col].transform(lambda x: x.fillna(x.median()))

# Confirm no missing values remain
print(df[impute_cols].isnull().sum())

Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
dtype: int64


In [10]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,121.677083,72.389323,29.089844,141.753906,32.434635,0.471876,33.240885,0.348958
std,3.369578,30.464161,12.106039,8.890820,89.100847,6.880498,0.331329,11.760232,0.476951
min,0.000000,44.000000,24.000000,7.000000,14.000000,18.200000,0.078000,21.000000,0.000000
25%,1.000000,99.750000,64.000000,25.000000,102.500000,27.500000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,28.000000,102.500000,32.050000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,169.500000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


Train/test split

Standard 80/20 split, stratified so both classes stay proportionally represented in both sets (important with imbalanced data like this to check the diabetes/no-diabetes ratio too).

In [11]:
from sklearn.model_selection import train_test_split

X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")
print(f"Train class balance:\n{y_train.value_counts(normalize=True)}")
print(f"Test class balance:\n{y_test.value_counts(normalize=True)}")

Train size: 614, Test size: 154
Train class balance:
Outcome
0    0.651466
1    0.348534
Name: proportion, dtype: float64
Test class balance:
Outcome
0    0.649351
1    0.350649
Name: proportion, dtype: float64


In [12]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()
gnb.fit(X_train, y_train)

print(f"Training accuracy: {gnb.score(X_train, y_train):.3f}")
print(f"Test accuracy: {gnb.score(X_test, y_test):.3f}")

Training accuracy: 0.783
Test accuracy: 0.727


In [13]:
# Weakly informative Normal-Inverse-Gamma prior, shared across classes,
# centred on the OVERALL feature means (not per-class, to avoid data leakage into the prior)
feature_cols = X_train.columns
overall_mean = X_train.mean()
overall_var = X_train.var()

mu0 = overall_mean          # prior mean
kappa0 = 1                  # prior pseudo-sample-size (weak = small)
alpha0 = 1                  # prior shape
beta0 = overall_var * (alpha0)   # prior scale, set so prior variance ≈ overall variance

print(mu0)

Pregnancies                   3.819218
Glucose                     121.713355
BloodPressure                72.145765
SkinThickness                29.017915
Insulin                     138.535831
BMI                          32.428176
DiabetesPedigreeFunction      0.477428
Age                          33.366450
dtype: float64


In [14]:
def fit_nig_posterior(X_train, y_train, feature_cols, mu0, kappa0, alpha0, beta0):
    posteriors = {}
    for cls in [0, 1]:
        X_cls = X_train[y_train == cls]
        n = len(X_cls)
        posteriors[cls] = {}
        for col in feature_cols:
            xbar = X_cls[col].mean()
            sse = ((X_cls[col] - xbar) ** 2).sum()

            kappa_n = kappa0 + n
            mu_n = (kappa0 * mu0[col] + n * xbar) / kappa_n
            alpha_n = alpha0 + n / 2
            beta_n = beta0[col] + 0.5 * sse + (kappa0 * n * (xbar - mu0[col])**2) / (2 * kappa_n)

            posteriors[cls][col] = {'mu_n': mu_n, 'kappa_n': kappa_n, 'alpha_n': alpha_n, 'beta_n': beta_n}
    return posteriors

nig_posteriors = fit_nig_posterior(X_train, y_train, feature_cols, mu0, kappa0, alpha0, beta0)
print(nig_posteriors[1]['Glucose'])  # sanity check: posterior for Glucose | diabetic

{'mu_n': np.float64(142.78471327929702), 'kappa_n': 215, 'alpha_n': 108.0, 'beta_n': np.float64(94238.95998819979)}


In [15]:
from scipy import stats

def student_t_logpdf(x, mu_n, kappa_n, alpha_n, beta_n):
    # Posterior predictive for a new observation is Student-t
    scale = np.sqrt(beta_n * (kappa_n + 1) / (alpha_n * kappa_n))
    df = 2 * alpha_n
    return stats.t.logpdf(x, df=df, loc=mu_n, scale=scale)

def predict_nig_nb(X, nig_posteriors, class_prior):
    preds = []
    for _, row in X.iterrows():
        log_scores = {}
        for cls in [0, 1]:
            log_score = np.log(class_prior[cls])
            for col in feature_cols:
                p = nig_posteriors[cls][col]
                log_score += student_t_logpdf(row[col], p['mu_n'], p['kappa_n'], p['alpha_n'], p['beta_n'])
            log_scores[cls] = log_score
        preds.append(1 if log_scores[1] > log_scores[0] else 0)
    return np.array(preds)

# Beta(1,1) prior on class probability -> posterior mean
n1 = (y_train == 1).sum()
n0 = (y_train == 0).sum()
class_prior = {1: (1 + n1) / (2 + n1 + n0), 0: (1 + n0) / (2 + n1 + n0)}
print(class_prior)

{1: np.float64(0.349025974025974), 0: np.float64(0.650974025974026)}


In [16]:
train_preds = predict_nig_nb(X_train, nig_posteriors, class_prior)
test_preds = predict_nig_nb(X_test, nig_posteriors, class_prior)

train_acc = (train_preds == y_train.values).mean()
test_acc = (test_preds == y_test.values).mean()

print(f"Manual NIG-Bayes NB — Train accuracy: {train_acc:.3f}, Test accuracy: {test_acc:.3f}")
print(f"sklearn GaussianNB — Train accuracy: 0.783, Test accuracy: 0.727")

Manual NIG-Bayes NB — Train accuracy: 0.785, Test accuracy: 0.727
sklearn GaussianNB — Train accuracy: 0.783, Test accuracy: 0.727


The manual conjugate NIG Naive Bayes achieves test accuracy (0.727) identical to sklearn's GaussianNB, with train accuracy only 0.002 higher. This confirms that with κ0=1 and 300+ observations per class, our weakly informative prior is quickly overwhelmed by the data — the posterior is data-dominated rather than prior-dominated. This is expected behavior for a well-specified weak prior and supports the robustness of our classification results to reasonable prior choices.

In [17]:
X_train.to_csv('/content/drive/MyDrive/X_train.csv', index=False)
X_test.to_csv('/content/drive/MyDrive/X_test.csv', index=False)
y_train.to_csv('/content/drive/MyDrive/y_train.csv', index=False)
y_test.to_csv('/content/drive/MyDrive/y_test.csv', index=False)
print("Saved train/test splits to shared Drive.")

Saved train/test splits to shared Drive.
